# Distribución de frecuencias — Versión 2: pandas en automático 🐼

En la notebook `DistribucionFrecuencia.ipynb` armamos la tabla **a mano**, como en el libro: nosotros elegimos 10 clases de ancho 5 (50-54, 55-59, ...), calculamos límites reales, fronteras y marcas de clase.

Aquí vamos a hacer **lo mismo, pero dejando que pandas decida los intervalos**. La idea es ver:

- Qué tan poquito código se necesita.
- De dónde salen esos intervalos "raros" que da pandas.
- Cómo se traducen los conceptos del libro (límites, fronteras, marcas de clase) a lo que nos da pandas.
- Qué cambia en las gráficas cuando los intervalos son otros.

Usamos los mismos datos: las 80 calificaciones de matemáticas.

# Estadística 

### Ejercicio de práctica libro de estadística. 

En la tabla siguiente se presentan las calificaciones finales que obtuvieron en matemáticas 80 alumnos de una
universidad.

68 84 75 82 68 90 62 88 76 93
73 79 88 73 60 93 71 59 85 75
61 65 75 87 74 62 95 78 63 72
66 78 82 75 94 77 69 74 68 60
96 78 89 61 75 95 60 79 83 71
79 62 67 97 78 85 76 65 71 75
65 80 73 57 88 78 62 76 53 74
86 67 73 81 72 63 76 75 85 77

Podemos manejar estos datos en un archivo csv y cargar los datos. O cargarlos directamente a una lista para posteriormente crear una Serie de pandas. En este ejercicio cargamos directamente los datos a una lista.

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

datos = [
    68, 84, 75, 82, 68, 90, 62, 88, 76, 93,
    73, 79, 88, 73, 60, 93, 71, 59, 85, 75,
    61, 65, 75, 87, 74, 62, 95, 78, 63, 72,
    66, 78, 82, 75, 94, 77, 69, 74, 68, 60,
    96, 78, 89, 61, 75, 95, 60, 79, 83, 71,
    79, 62, 67, 97, 78, 85, 76, 65, 71, 75,
    65, 80, 73, 57, 88, 78, 62, 76, 53, 74,
    86, 67, 73, 81, 72, 63, 76, 75, 85, 77
]

print("Datos capturados en la lista:")
print(datos)
print("---------------------------------")

# Creamos una serie de Pandas a partir de la lista de datos
calificaciones = pd.Series(datos)
print("Datos capturados en la serie de Pandas:")
print(calificaciones)
print("---------------------------------")


Podemos obtener un resumen de los datos ejecutando la función describe() de pandas:

In [ ]:
print("Resumen de datos")
print(calificaciones.describe())
print("-------------------------------------------------")

# Paso 1: La distribución en una sola línea

Pandas tiene `value_counts(bins=10)`: le dices "hazme 10 intervalos" y él solito calcula dónde cortar y cuenta cuántos datos caen en cada uno. `sort_index()` es nada más para que salgan ordenados de menor a mayor.

In [ ]:
frecuencias_auto = calificaciones.value_counts(bins=10).sort_index()
frecuencias_auto

¡Y ya! 😮 Eso es una distribución de frecuencias. Pero hay que entender qué nos está dando, porque se ve un poco rara.

## ¿Cómo lee uno `(57.4, 61.8]`?

Fíjate en los signos: abre con **paréntesis** `(` y cierra con **corchete** `]`.

- `(` significa **"sin incluir"**: el 57.4 **no** entra en este intervalo.
- `]` significa **"incluyendo"**: el 61.8 **sí** entra.

O sea, `(57.4, 61.8]` quiere decir: **mayor que 57.4 y menor o igual que 61.8**.

Esto es importante, porque el 75, por ejemplo, cae en `(70.6, 75.0]` y **no** en `(75.0, 79.4]`.

> 💡 Gracias a esto, un intervalo termina **exactamente** donde empieza el siguiente y ningún dato puede caer en dos intervalos al mismo tiempo. ¡Es la misma idea que los límites reales del libro! Pandas resuelve el problema de los "huecos" con paréntesis y corchetes en lugar de restar 0.5.

## ¿De dónde salen esos números?

Pandas hace esto:

1. Toma el **dato menor (53)** y el **dato mayor (97)**.
2. Reparte ese tramo en 10 partes iguales: $\frac{97 - 53}{10} = 4.4$. Por eso todos los intervalos miden **4.4**.
3. Para que el 53 no se quede fuera (el primer intervalo empieza con paréntesis, o sea "sin incluir"), recorre el inicio un poquitito hacia abajo: un 0.1 % del rango ($44 \times 0.001 = 0.044$). Por eso el primer intervalo empieza en **52.956** en lugar de 53.

Comprobémoslo:

In [ ]:
n = len(calificaciones)
dato_menor = calificaciones.min()
dato_mayor = calificaciones.max()
rango = dato_mayor - dato_menor
k = 10

ancho = rango / k
print("Rango:", rango)
print("Ancho de cada intervalo:", ancho)
print("Inicio del primer intervalo:", dato_menor - rango * 0.001)

# Paso 2: Sacar las fronteras de clase con `pd.cut`

`value_counts(bins=10)` es cómodo, pero no nos da directamente los puntos de corte. Para eso está su primo **`pd.cut`**: le pasas los datos y el número de intervalos, y con `retbins=True` también te regresa las **fronteras** (los puntos donde cortó).

`pd.cut` hace algo muy sencillo: a cada calificación le pone una "etiqueta" con el intervalo al que pertenece.

In [ ]:
intervalos, fronteras = pd.cut(calificaciones, bins=k, retbins=True)

print("Fronteras de clase:")
print(fronteras)
print()
print("Las primeras 5 calificaciones y su intervalo:")
print(pd.DataFrame({"calificación": calificaciones, "intervalo": intervalos}).head())

Son 11 fronteras para 10 clases (10 clases necesitan 11 "paredes"), igualito que en el libro.

> 📝 Notarás que aquí sale `52.956` y en `value_counts` salía `52.955000000000005`. Es un detallito de cómo cada función redondea las etiquetas; los intervalos son los mismos y los conteos también. Usaremos las de `pd.cut`, que se ven más limpias.

# Paso 3: Límites y marcas de clase

Cada intervalo de pandas es un objeto `Interval` que ya trae todo lo que necesitamos:

| Concepto del libro | En pandas |
|---|---|
| Límite inferior | `.left` |
| Límite superior | `.right` |
| Marca de clase (punto medio) | `.mid` |
| Ancho de clase | `.length` |

Contamos cuántos datos hay en cada intervalo con `value_counts(sort=False)` (`sort=False` para que respete el orden de los intervalos y no los ordene por frecuencia):

In [ ]:
frecuencias = intervalos.value_counts(sort=False)
clases = pd.IntervalIndex(frecuencias.index)      # los intervalos

print("Límites inferiores:", clases.left.tolist())
print("Límites superiores:", clases.right.tolist())
print("Marcas de clase:   ", clases.mid.round(2).tolist())
print("Anchos:            ", clases.length.round(2).tolist())
print("Frecuencias:       ", frecuencias.tolist())
print("Suma:", frecuencias.sum())

Algunas cosas que notar:

- **Límites reales:** aquí no hace falta calcularlos aparte. Como los intervalos de pandas ya son **continuos** (el final de uno es el inicio del otro), los límites de pandas **ya son** los límites reales / fronteras. No hay que restar 0.5 a nada.
- **Marcas de clase:** son el punto medio de cada intervalo, por ejemplo $\frac{57.4 + 61.8}{2} = 59.6$. La primera (55.18) se ve un poco chueca porque ese intervalo empieza en 52.956 y por eso mide 4.444 en vez de 4.4.
- La suma da **80**. ✅ No se perdió nadie.

# Paso 4: La tabla completa

Igual que en la versión del libro, armamos la tabla con frecuencia relativa, porcentaje y acumuladas. Aquí casi todo se hace con métodos de pandas:

- `normalize=True` en `value_counts` → frecuencia relativa directo.
- `cumsum()` → suma acumulada.

In [ ]:
tabla = pd.DataFrame({
    "Intervalo": clases.astype(str),
    "Lím. inf": clases.left,
    "Lím. sup": clases.right,
    "Marca de clase": clases.mid.round(2),
    "f": frecuencias.values,
})

tabla["fr"] = intervalos.value_counts(sort=False, normalize=True).values
tabla["%"] = tabla["fr"] * 100
tabla["F"] = tabla["f"].cumsum()
tabla["Fr"] = tabla["F"] / n

tabla

**¿Cómo se lee?**

- La clase con más alumnos (la **clase modal**) es **(70.6, 75.0]** con **19 alumnos** (23.75 %).
- En la fila **(66.2, 70.6]**, $F = 24$: **24 alumnos sacaron 70.6 o menos**, o sea, 70 o menos.

👀 **Ojo:** en la versión del libro la clase modal era **75 - 79** con 21 alumnos. Aquí es **(70.6, 75]**. ¿Por qué? Porque **7 alumnos sacaron exactamente 75**, y pandas los mete en el intervalo que **termina** en 75, mientras que el libro los mete en la clase que **empieza** en 75. ¡Los mismos datos, otros cortes, otra historia! Esto lo vamos a ver clarito en las gráficas.

In [ ]:
print("¿Cuántos sacaron exactamente 75?", (calificaciones == 75).sum())

---

# Gráficas, versión pandas

Las series de pandas tienen el atributo `.plot`, que por dentro usa matplotlib. Así que muchas gráficas salen en una sola línea: `calificaciones.plot.box()`, `calificaciones.plot.kde()`...

## 1. Histograma

Aquí hay una trampa en la que es muy fácil caer, así que vamos con calma. 🪤

Lo natural sería hacer `calificaciones.plot.hist(bins=fronteras)`. Pero el histograma de matplotlib (y numpy) usa intervalos **al revés** que pandas:

- **pandas** `pd.cut`: `(a, b]` → incluye el de la **derecha**.
- **matplotlib / numpy** `hist`: `[a, b)` → incluye el de la **izquierda**.

En la mayoría de los datos esto no importa, pero aquí **75 es una frontera** y hay 7 alumnos con 75. Pandas los mete en (70.6, 75] y matplotlib los mandaría a [75, 79.4). ¡La gráfica no coincidiría con la tabla! Mira:

In [ ]:
conteo_matplotlib, _ = np.histogram(calificaciones, bins=fronteras)   # lo que contaría plt.hist

print("Pandas (tabla):        ", frecuencias.tolist())
print("Matplotlib (plt.hist): ", conteo_matplotlib.tolist())

¿Ves la diferencia en las clases 5 y 6? (19 y 14 contra 12 y 21). Por eso, para que el histograma diga **exactamente** lo mismo que nuestra tabla, lo dibujamos con barras (`plt.bar`) usando las frecuencias que ya contó pandas:

- Cada barra empieza en el **límite inferior** (`clases.left`) → `align="edge"`.
- Su ancho es el **ancho de clase** (`clases.length`), así quedan pegaditas.
- Su altura es la **frecuencia**.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(clases.left, frecuencias, width=clases.length, align="edge", edgecolor="black", color="skyblue")

plt.title("Histograma de calificaciones (intervalos automáticos de pandas)")
plt.xlabel("Calificaciones")
plt.ylabel("Frecuencia")
plt.xticks(fronteras.round(1))
plt.show()

> 💡 Moraleja: cuando un dato cae justo en una frontera, **quién se lo queda depende de la herramienta**. Por eso el libro usa fronteras como 74.5, que ninguna calificación entera puede tocar. ¡Otro punto a favor de los límites reales!

**¿Qué vemos?** El pico está en (70.6, 75], y se ve una forma más "irregular" que la del libro: hay un hoyito en (66.2, 70.6] y la última barra (92.6, 97] está más alta que la de antes.

## 2. Polígono de frecuencias

Igual que antes: un punto en cada **marca de clase** a la altura de su frecuencia, y agregamos una clase "fantasma" con frecuencia 0 a cada lado (a una distancia de 4.4) para que el polígono baje al piso.

In [ ]:
marcas = clases.mid.tolist()
x_poligono = [marcas[0] - ancho] + marcas + [marcas[-1] + ancho]
y_poligono = [0] + frecuencias.tolist() + [0]

plt.figure(figsize=(9, 5))
plt.bar(clases.left, frecuencias, width=clases.length, align="edge", edgecolor="black", color="skyblue", label="Histograma")
plt.plot(x_poligono, y_poligono, marker="o", color="red", label="Polígono de frecuencias")

plt.title("Histograma + Polígono de frecuencias")
plt.xlabel("Calificaciones")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

## 3. Ojivas

Para las ojivas usamos las **fronteras** en el eje X y las frecuencias acumuladas en el eje Y:

- **"Menor que":** empieza en 0 y va subiendo hasta 80.
- **"Mayor que":** empieza en 80 y va bajando hasta 0.

In [ ]:
acumulada_menor = [0] + tabla["F"].tolist()
acumulada_mayor = [n - a for a in acumulada_menor]

plt.figure(figsize=(9, 5))
plt.plot(fronteras, acumulada_menor, marker="o", color="green", label='Ojiva "menor que"')
plt.plot(fronteras, acumulada_mayor, marker="s", color="purple", label='Ojiva "mayor que"')

plt.title("Ojivas (intervalos automáticos de pandas)")
plt.xlabel("Fronteras de clase")
plt.ylabel("Frecuencia acumulada")
plt.xticks(fronteras.round(1))
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

Las dos ojivas se cruzan alrededor de **74.3**, cerquita de la mediana real, que es **75**.

¿Por qué no da exacto? Porque al agrupar **perdemos un poco de información**: la ojiva supone que los datos están repartidos parejito dentro de cada intervalo, y en realidad hay un montón de 75 amontonados justo en la orilla de (70.6, 75]. Con la versión del libro el cruce quedaba más cerca de 75.

In [ ]:
print("Mediana real:", calificaciones.median())

## 4. Diagrama de caja (boxplot)

El boxplot **no depende de los intervalos**: usa los datos originales para sacar los cuartiles. Así que debe salir **idéntico** al de la versión del libro. En pandas es una línea:

In [ ]:
q1 = calificaciones.quantile(0.25)
q3 = calificaciones.quantile(0.75)
print("Q1:", q1, "| Mediana:", calificaciones.median(), "| Q3:", q3, "| RIC:", q3 - q1)

calificaciones.plot.box(vert=False, figsize=(9, 3))
plt.title("Diagrama de caja de las calificaciones")
plt.xlabel("Calificaciones")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.show()

Exactamente igual que antes: la caja de 67.75 a 82, mediana en 75 y sin datos atípicos. 👍 Esto es una ventaja del boxplot: **no importa cómo agrupes, siempre sale igual**.

## 5. Curva de frecuencias

### Opción A: suavizar el polígono
Igual que en la versión del libro, con `make_interp_spline`:

In [ ]:
from scipy.interpolate import make_interp_spline

x_suave = np.linspace(min(x_poligono), max(x_poligono), 300)
curva = make_interp_spline(x_poligono, y_poligono, k=3)
y_suave = np.clip(curva(x_suave), 0, None)   # que no baje de 0

plt.figure(figsize=(9, 5))
plt.bar(clases.left, frecuencias, width=clases.length, align="edge", edgecolor="black", color="skyblue", alpha=0.4, label="Histograma")
plt.plot(x_poligono, y_poligono, "--", color="gray", label="Polígono de frecuencias")
plt.plot(x_suave, y_suave, color="orange", linewidth=3, label="Curva de frecuencias")

plt.title("Curva de frecuencias (polígono suavizado)")
plt.xlabel("Calificaciones")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

### Opción B: la curva automática de pandas (KDE)

`calificaciones.plot.kde()` dibuja la curva **directo desde los datos originales**, sin usar los intervalos. Por eso sale suavecita y no tiene los "brincos" del polígono. Como usa **densidad** (el área bajo la curva vale 1), las barras del histograma también las pasamos a densidad para que estén en la misma escala: $	ext{densidad} = rac{f_r}{	ext{ancho}}$.

In [ ]:
densidad = tabla["fr"] / clases.length

plt.figure(figsize=(9, 5))
plt.bar(clases.left, densidad, width=clases.length, align="edge", edgecolor="black", color="skyblue", alpha=0.4)
calificaciones.plot.kde(color="orange", linewidth=3)

plt.title("Curva de frecuencias con KDE de pandas")
plt.xlabel("Calificaciones")
plt.ylabel("Densidad")
plt.show()

La KDE sale igual que en la versión del libro (porque tampoco depende de los intervalos) y otra vez se ve la forma de **campana**, más o menos simétrica, con el centro cerca de 75.

---

# Libro vs. pandas: cara a cara 🥊

Pongamos los dos histogramas juntos para ver cómo cambia el dibujo con los mismos 80 datos:

In [ ]:
fronteras_libro = np.arange(49.5, 100, 5)     # 49.5, 54.5, ..., 99.5

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

ejes[0].hist(calificaciones, bins=fronteras_libro, edgecolor="black", color="lightgreen")
ejes[0].set_title("Libro: 10 clases de ancho 5")
ejes[0].set_xlabel("Calificaciones")
ejes[0].set_ylabel("Frecuencia")

ejes[1].bar(clases.left, frecuencias, width=clases.length, align="edge", edgecolor="black", color="skyblue")
ejes[1].set_title("Pandas: 10 clases de ancho 4.4")
ejes[1].set_xlabel("Calificaciones")

plt.show()

Los dos se parecen en lo general (campana con el centro por ahí del 75), pero los detalles cambian: dónde está el pico, qué tan alto es, si hay hoyitos... **El histograma depende de cómo elijas los intervalos.** Por eso siempre conviene probar con distintos cortes antes de sacar conclusiones fuertes.

> 🔧 **Truco:** `pd.cut` también acepta tus propias fronteras. Si le pasas las del libro, obtienes exactamente la tabla del libro, pero con el poquito código de pandas:

In [ ]:
intervalos_libro = pd.cut(calificaciones, bins=fronteras_libro)
intervalos_libro.value_counts(sort=False)

¡Las mismas frecuencias de la notebook 1 (1, 2, 11, 10, 12, 21, 6, 9, 4, 4)! 🎉

## Resumen: ¿cuál uso?

| | A mano (libro) | Pandas automático |
|---|---|---|
| Intervalos | Redondos y fáciles de leer (50-54) | Con decimales (57.4, 61.8] |
| Huecos entre clases | Se arreglan con límites reales (±0.5) | No hay: usa `(` y `]` |
| Marcas de clase | Números bonitos (52, 57...) | Decimales (59.6, 64...) |
| Código | Más largo | Una o dos líneas |
| Ideal para | Reportes, tareas, presentar resultados | Explorar datos rápido |

**Mi conclusión:** la versión automática es perfecta para **echar un primer vistazo** a los datos en segundos (justo lo que se hace en ciencia de datos al empezar un análisis). Pero cuando vas a **presentar** los resultados a alguien, es mejor elegir intervalos redondos que tengan sentido, y con `pd.cut(bins=...)` puedes tener lo mejor de los dos mundos: intervalos bonitos con poquito código.

Y lo más importante: aunque pandas lo haga todo solito, **hay que entender los conceptos** (límites, fronteras, marcas de clase) para saber leer lo que nos da y no tragarnos un `(52.955000000000005, 57.4]` sin saber qué significa. 😉